<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/Step8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FAIML_DANIELE_drive//MaskArchitectureAnomaly_CourseProject
!pip install -r eomt/requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject


In [2]:


import os, glob

DATA_ROOT = "data/Validation_Dataset"
print(os.listdir(DATA_ROOT))

sample_images = glob.glob(f"{DATA_ROOT}/RoadAnomaly21/images/*")
print(len(sample_images), sample_images[:3])

['RoadAnomaly21', 'RoadAnomaly', 'RoadObsticle21', 'fs_static', 'FS_LostFound_full', '.DS_Store']
10 ['data/Validation_Dataset/RoadAnomaly21/images/8.png', 'data/Validation_Dataset/RoadAnomaly21/images/9.png', 'data/Validation_Dataset/RoadAnomaly21/images/4.png']


In [3]:
import sys, yaml
sys.path.append("eomt")

config_path = "eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config["trainer"]["logger"]["init_args"]["name"]

'cityscapes_semantic_eomt_base_640'

In [4]:
from huggingface_hub import hf_hub_download

name = config["trainer"]["logger"]["init_args"]["name"]

ckpt_path = hf_hub_download(
    repo_id=f"S362484/{name}",
    filename="eomt_cityscapes.bin",
)

print(ckpt_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/root/.cache/huggingface/hub/models--S362484--cityscapes_semantic_eomt_base_640/snapshots/0a0d8a920f846a2ba1347377f55a7fc2daa17a50/eomt_cityscapes.bin


In [5]:
import os, sys, yaml, warnings, importlib
import torch
from torch.nn import functional as F
from lightning import seed_everything

seed_everything(0, verbose=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PROJECT_ROOT = "/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject"
EOMT_ROOT = f"{PROJECT_ROOT}/eomt"

%cd {EOMT_ROOT}

if EOMT_ROOT not in sys.path:
    sys.path.append(EOMT_ROOT)

config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

img_size = (1024, 1024)
num_classes = 19

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)


encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)

encoder = encoder_cls(
    img_size=img_size,
    **encoder_cfg.get("init_args", {})
)


network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

network_kwargs = {
    k: v for k, v in network_cfg["init_args"].items()
    if k != "encoder"
}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes,
    encoder=encoder,
    **network_kwargs,
)


lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)

model_kwargs = {
    k: v for k, v in config["model"]["init_args"].items()
    if k != "network"
}

model = lit_cls(
    img_size=img_size,
    num_classes=num_classes,
    network=network,
    **model_kwargs,
).eval().to(device)


state_dict = torch.load(
    ckpt_path,
    map_location=device,
    weights_only=False,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Model loaded")
print("missing keys:", len(missing))
print("unexpected keys:", len(unexpected))

device: cuda
/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject/eomt
Model loaded
missing keys: 0
unexpected keys: 0


In [9]:
%cd /content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject

!python eval/eval_eomt.py \
  --datasets RoadAnomaly RoadAnomaly21 fs_static LostFound RoadObsticle21 \
  --methods msp entropy maxlogit rba

/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
Loaded EoMT checkpoint: /root/.cache/huggingface/hub/models--S362484--cityscapes_semantic_eomt_base_640/snapshots/0a0d8a920f846a2ba1347377f55a7fc2daa17a50/eomt_cityscapes.bin
Missing keys: 0 | Unexpected keys: 0

========== DATASET: RoadAnomaly ==========
data/Validation_Dataset/RoadAnomaly/images/0.jpg
data/Validation_Dataset/RoadAnomaly/images/1.jpg
data/Validation_Dataset/RoadAnomaly/images/10.jpg
data/Validation_Dataset/RoadAnomaly/images/11.jpg
data/Validation_Dataset/RoadAnomaly/images/12.jpg
data/Validation_Dataset/RoadAnomaly/images/13.jpg
data/Validation_Dataset/RoadAnomaly/images/14.jpg
data/Validation_Dataset/RoadAnomaly/images/15.jpg


In [10]:
with open("results_eomt.txt", "r") as f:
    print(f.read())


EoMT anomaly evaluation
     RoadAnomaly       msp  AUPRC: 75.1655  FPR@TPR95: 19.0710
     RoadAnomaly   entropy  AUPRC: 51.5438  FPR@TPR95: 44.0736
     RoadAnomaly  maxlogit  AUPRC: 70.9057  FPR@TPR95: 44.8832
   RoadAnomaly21       msp  AUPRC: 73.8746  FPR@TPR95: 33.5757
   RoadAnomaly21   entropy  AUPRC: 67.5200  FPR@TPR95: 26.1103
   RoadAnomaly21  maxlogit  AUPRC: 72.8476  FPR@TPR95: 20.9331
       fs_static       msp  AUPRC: 54.5348  FPR@TPR95: 34.8135
       fs_static   entropy  AUPRC: 11.8682  FPR@TPR95: 56.8836
       fs_static  maxlogit  AUPRC: 53.4809  FPR@TPR95: 77.9201

EoMT anomaly evaluation
     RoadAnomaly       msp  AUPRC: 75.1655  FPR@TPR95: 19.0710
     RoadAnomaly   entropy  AUPRC: 51.5438  FPR@TPR95: 44.0736
     RoadAnomaly  maxlogit  AUPRC: 70.9057  FPR@TPR95: 44.8832
   RoadAnomaly21       msp  AUPRC: 73.8746  FPR@TPR95: 33.5757
   RoadAnomaly21   entropy  AUPRC: 67.5200  FPR@TPR95: 26.1103
   RoadAnomaly21  maxlogit  AUPRC: 72.8476  FPR@TPR95: 20.9331
     